# Session 4: Data Cleaning with Pandas
## Handling Nulls, Duplicates & Data Type Conversions

**Modern Data Engineering Course** — Week 2  
**Instructors:** Ameer Ul Islam

---

## 1. Why is Real-World Data Messy?

Data engineers spend **60-80% of their time cleaning data.** Common issues include:

- **Missing values** — sensors fail, users skip fields, APIs return nulls
- **Duplicate records** — double submissions, merge errors, retry logic
- **Wrong data types** — numbers stored as strings, dates as text
- **Inconsistent formats** — "NY" vs "New York" vs "new york"
- **Outliers & errors** — age = 999, price = -50

Cleaning data properly = trustworthy analytics & reliable pipelines.

## 2. Our Messy Practice Dataset

We'll work with this intentionally messy e-commerce dataset throughout the session.

In [142]:
import pandas as pd
import numpy as np

df = pd.read_csv("messy_orders.csv")
print(df)

   order_id      customer product   amount  order_date
0     101.0   Elif Yılmaz  Laptop  85000.0  2024-01-15
1     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
2     103.0           NaN  Tablet  32000.0      Jan 17
3     104.0   Elif Yılmaz  Laptop      NaN  2024-01-15
4     105.0   Burak Demir     NaN  12000.0  2024-01-18
5     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
6     106.0  Zeynep Çelik   Mouse   2500.0  2024/01/19
7       NaN           NaN     NaN      NaN         NaN


In [143]:
# Let's explore what's wrong
print("=== Shape ===")
print(df.shape)
print()

print("=== Data Types ===")
print(df.dtypes)
print()

print("=== Info ===")
df.info()

=== Shape ===
(8, 5)

=== Data Types ===
order_id      float64
customer       object
product        object
amount        float64
order_date     object
dtype: object

=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    7 non-null      float64
 1   customer    6 non-null      object 
 2   product     6 non-null      object 
 3   amount      6 non-null      float64
 4   order_date  7 non-null      object 
dtypes: float64(2), object(3)
memory usage: 452.0+ bytes


### Issues to find & fix:
1. **Nulls** — `None` and `NaN` values in customer, product, amount
2. **Duplicate** — order_id 102 is repeated
3. **Wrong type** — amount is a string (`object`), not a number
4. **Bad dates** — mixed formats: "2024-01-15", "Jan 17", "2024/01/19"


---
## 3. Detecting Missing Data

Before you clean, you need to know what's dirty.

In [144]:
# Check for nulls — boolean mask (True = null)
print("=== Null Mask ===")
print(df.isnull())

=== Null Mask ===
   order_id  customer  product  amount  order_date
0     False     False    False   False       False
1     False     False    False   False       False
2     False      True    False   False       False
3     False     False    False    True       False
4     False     False     True   False       False
5     False     False    False   False       False
6     False     False    False   False       False
7      True      True     True    True        True


In [145]:
# Count nulls per column — the most useful check!
print("=== Null Count Per Column ===")
print(df.isnull().sum())

=== Null Count Per Column ===
order_id      1
customer      2
product       2
amount        2
order_date    1
dtype: int64


In [146]:
# Percentage of nulls per column
print("=== Null Percentage ===")
print(len(df)) #counts rows
null_pct = (df.isnull().sum() / len(df)) * 100
print(null_pct.round(1))

=== Null Percentage ===
8
order_id      12.5
customer      25.0
product       25.0
amount        25.0
order_date    12.5
dtype: float64


In [147]:
# Check a specific column
print(df["customer"])
print(df["customer"].isnull())
print(f'Customer nulls: {df["customer"].isnull().sum()}')
print(f'Product nulls:  {df["product"].isnull().sum()}')
print(f'Amount nulls:   {df["amount"].isnull().sum()}')
print(f'Total nulls:    {df.isnull().sum().sum()}')

0     Elif Yılmaz
1     Mehmet Kaya
2             NaN
3     Elif Yılmaz
4     Burak Demir
5     Mehmet Kaya
6    Zeynep Çelik
7             NaN
Name: customer, dtype: object
0    False
1    False
2     True
3    False
4    False
5    False
6    False
7     True
Name: customer, dtype: bool
Customer nulls: 2
Product nulls:  2
Amount nulls:   2
Total nulls:    8


---
## 4. Handling Nulls: Dropping Rows

Sometimes the simplest approach is to remove incomplete rows.

In [148]:
# Drop rows with ANY null value
print(df)
df_dropped_any = df.dropna()
print(f"Original: {len(df)} rows")
print(f"After dropna(): {len(df_dropped_any)} rows")
print(f"Lost: {len(df) - len(df_dropped_any)} rows")
print()
print(df_dropped_any)

   order_id      customer product   amount  order_date
0     101.0   Elif Yılmaz  Laptop  85000.0  2024-01-15
1     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
2     103.0           NaN  Tablet  32000.0      Jan 17
3     104.0   Elif Yılmaz  Laptop      NaN  2024-01-15
4     105.0   Burak Demir     NaN  12000.0  2024-01-18
5     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
6     106.0  Zeynep Çelik   Mouse   2500.0  2024/01/19
7       NaN           NaN     NaN      NaN         NaN
Original: 8 rows
After dropna(): 4 rows
Lost: 4 rows

   order_id      customer product   amount  order_date
0     101.0   Elif Yılmaz  Laptop  85000.0  2024-01-15
1     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
5     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
6     106.0  Zeynep Çelik   Mouse   2500.0  2024/01/19


In [149]:
# Drop rows where ALL values are null (won't drop anything here)
df_dropped_all = df.dropna(how="all")
print(f"After dropna(how='all'): {len(df_dropped_all)} rows")

After dropna(how='all'): 7 rows


In [150]:
# Drop rows with nulls only in specific columns
df_dropped_subset = df.dropna(subset=["customer", "amount"])
print(f"After dropna(subset=['customer','amount']): {len(df_dropped_subset)} rows")
print()
print(df_dropped_subset)

After dropna(subset=['customer','amount']): 5 rows

   order_id      customer product   amount  order_date
0     101.0   Elif Yılmaz  Laptop  85000.0  2024-01-15
1     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
4     105.0   Burak Demir     NaN  12000.0  2024-01-18
5     102.0   Mehmet Kaya   Phone  55000.0  2024-01-16
6     106.0  Zeynep Çelik   Mouse   2500.0  2024/01/19


In [151]:
# Keep rows with at least N non-null values
df_thresh = df.dropna(thresh=4)  # keep rows with at least 5 non-null values
print(f"After dropna(thresh=4): {len(df_thresh)} rows")

After dropna(thresh=4): 7 rows


---
## 5. Handling Nulls: Filling Values

Replace missing values with something meaningful.

In [152]:
# Fill with a constant value
df_filled = df.copy()
df_filled["customer"] = df_filled["customer"].fillna("Unknown")
df_filled["order_id"] = df_filled["order_id"].fillna(999)
print("=== After filling customer with 'Unknown' ===")
print(df_filled[["order_id", "customer"]])

=== After filling customer with 'Unknown' ===
   order_id      customer
0     101.0   Elif Yılmaz
1     102.0   Mehmet Kaya
2     103.0       Unknown
3     104.0   Elif Yılmaz
4     105.0   Burak Demir
5     102.0   Mehmet Kaya
6     106.0  Zeynep Çelik
7     999.0       Unknown


In [153]:
# Fill with the most frequent value (mode)
print(df["product"])
print(df["product"].mode()) # it returns the most frequent value(s)
most_common_product = df["product"].mode()[1]  # mode() returns a Series
print(f"Most common product: {most_common_product}")

df_filled["product"] = df_filled["product"].fillna(most_common_product)
print()
print("=== After filling product with mode ===")
print(df_filled[["order_id", "product"]])

0    Laptop
1     Phone
2    Tablet
3    Laptop
4       NaN
5     Phone
6     Mouse
7       NaN
Name: product, dtype: object
0    Laptop
1     Phone
Name: product, dtype: object
Most common product: Phone

=== After filling product with mode ===
   order_id product
0     101.0  Laptop
1     102.0   Phone
2     103.0  Tablet
3     104.0  Laptop
4     105.0   Phone
5     102.0   Phone
6     106.0   Mouse
7     999.0   Phone


In [163]:
# For numeric columns — fill with mean or median
# First we need to convert amount to numeric (we'll cover this properly later)
data = {
    "order_id":   [101, 102, 103, 104, 105, 102, 106],
    "customer":   ["Rahim", "Fatima", None, "Rahim", "Karim", "Fatima", "Ayesha"],
    "product":    ["Laptop", "Phone", "Tablet", "Laptop", None, "Phone", "Mouse"],
    "amount":     ["85000", "55000", "32000", np.nan, "12000", "55000", "2500"],
    "order_date": ["2024-01-15", "2024-01-16", "Jan 17", "2024-01-15",
                   "2024-01-18", "2024-01-16", "2024/01/19"],
}

df_new = pd.DataFrame(data)
# print(df_new)
# print(df_new.info())
amounts_numeric = pd.to_numeric(df_new["amount"], errors="coerce")

print(f"Mean amount:   {amounts_numeric.mean():,.0f}")
print(f"Median amount: {amounts_numeric.median():,.0f}")

# # Fill with median (better for skewed data)
df_new["amount"] = amounts_numeric.fillna(amounts_numeric.median())
print()
print("=== After filling amount with median ===")
print(df_new[["order_id", "amount"]])

Mean amount:   40,250
Median amount: 43,500

=== After filling amount with median ===
   order_id   amount
0       101  85000.0
1       102  55000.0
2       103  32000.0
3       104  43500.0
4       105  12000.0
5       102  55000.0
6       106   2500.0


In [164]:
# Forward fill — use previous row's value
df_ffill = df.copy()
df_ffill["product"] = df_ffill["product"].ffill()
print("=== Forward Fill ===")
print(df_ffill[["order_id", "product"]])

=== Forward Fill ===
   order_id product
0     101.0  Laptop
1     102.0   Phone
2     103.0  Tablet
3     104.0  Laptop
4     105.0  Laptop
5     102.0   Phone
6     106.0   Mouse
7       NaN   Mouse


In [165]:
# Backward fill — use the NEXT row's value
df_bfill = df.copy()
df_bfill["product"] = df_bfill["product"].bfill()
print("=== Backward Fill ===")
print(df_bfill[["order_id", "product"]])

=== Backward Fill ===
   order_id product
0     101.0  Laptop
1     102.0   Phone
2     103.0  Tablet
3     104.0  Laptop
4     105.0   Phone
5     102.0   Phone
6     106.0   Mouse
7       NaN     NaN


### Which Fill Strategy to Use?

| Data Type | Strategy | When to Use |
|---|---|---|
| **Numeric** | Mean / Median | Normal distribution → mean. Skewed → median |
| **Categorical** | Mode / "Unknown" | Use most frequent value or a placeholder |
| **Time Series** | Forward / Backward Fill | Data has time ordering — carry previous value |
| **Business Logic** | Custom Rules | Domain-specific: e.g. missing quantity = 0 |

> 💡 **Always check `df.describe()` after filling — make sure stats still make sense!**


---
## Hands-On Exercise 1: Find & Fix Missing Data

Using our messy dataset, do the following:

1. Check nulls with `df.isnull().sum()`
2. Fill customer nulls with "Unknown"
3. Fill product nulls with the mode
4. Drop rows where amount is null
5. Verify: `df.isnull().sum()` should show all zeros


In [ ]:
# ✅ Exercise 1 Solution

df = pd.read_csv("messy_orders.csv")
print("=== Before Cleaning ===")
print(df.isnull().sum())
print()

# Fill customer
df["customer"] = df["customer"].fillna("Unknown")

# Fill product with mode
df["product"] = df["product"].fillna(df["product"].mode()[0])

# Convert amount to numeric first, then drop nulls
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df = df.dropna(subset=["amount"])

print("=== After Cleaning ===")
print(df.isnull().sum())
print()
print(f"Rows remaining: {len(df)}")
print(df)

---
## ☕ Break Time! (10 minutes)

Stretch, grab a drink, and come back refreshed!

---

## 6. Finding Duplicates

Duplicates silently inflate your metrics — always check!

In [ ]:
# Recreate our messy data
df = pd.read_csv("messy_orders.csv")

# Check for duplicates
print(f"Total duplicate rows: {df.duplicated().sum()}")
print()

# View the duplicate rows (keep=False shows ALL copies, not just the extra)
print("=== Duplicate Rows ===")
print(df[df.duplicated(keep=False)])

In [ ]:
# Check duplicates based on specific columns
print("=== Rows with duplicate order_id ===")
print(df[df.duplicated(subset=["order_id"], keep=False)])

## 7. Removing Duplicates

In [ ]:
# Remove exact duplicate rows (keep first occurrence)
df_clean = df.drop_duplicates()
print(f"Before: {len(df)} rows")
print(f"After drop_duplicates(): {len(df_clean)} rows")

In [ ]:
# Remove based on specific columns
df_clean = df.drop_duplicates(subset=["order_id"])
print(f"After drop_duplicates(subset=['order_id']): {len(df_clean)} rows")
print()
print(df_clean)

In [ ]:
# Keep last occurrence instead of first
df_clean_last = df.drop_duplicates(subset=["order_id"], keep="last")
print(f"Keeping last: {len(df_clean_last)} rows")

# Verify — always print before/after!
print(f'\nBefore: {len(df)} rows → After: {len(df_clean)} rows → Removed: {len(df) - len(df_clean)}')

### 🔑 Data Engineering Best Practice

> Always check **WHY** duplicates exist before removing them. Sometimes they indicate a bug in your data pipeline that needs fixing at the source, not just cleaning at the end.


---
## 8. Data Type Conversions

Wrong data types = broken calculations, failed joins, and slow queries.

In [ ]:
# Check current types
df = pd.DataFrame(data)
df = df.drop_duplicates(subset=["order_id"])  # clean dupes first

print("=== Current Data Types ===")
print(df.dtypes)
print()
print("Notice: 'amount' is 'object' (string), not a number!")
print("Notice: 'order_date' is 'object' (string), not datetime!")

### 8.1 Converting to Numeric Types

In [ ]:
# String to float — the simple way
# df["amount"] = df["amount"].astype(float)  # ⚠️ This crashes if there's a non-numeric value!

# Safe conversion — errors become NaN instead of crashing
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
print("=== After to_numeric ===")
print(df.dtypes)
print()
print(df[["order_id", "amount"]])

In [ ]:
# Now we can do math!
print(f"Total sales:   {df['amount'].sum():,.0f} BDT")
print(f"Average order: {df['amount'].mean():,.0f} BDT")
print(f"Max order:     {df['amount'].max():,.0f} BDT")

In [ ]:
# Nullable integer type — allows NaN + int together
# Normal int can't have NaN values. Capital-I "Int64" can!
df["order_id_int"] = df["order_id"].astype("Int64")  # Capital I!
print(df.dtypes)

### 8.2 Converting to DateTime

In [ ]:
# Convert date strings to proper datetime
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
# errors="coerce" turns unparseable dates into NaT (Not a Time)

print("=== After to_datetime ===")
print(df.dtypes)
print()
print(df[["order_id", "order_date"]])
print()
print("Notice: 'Jan 17' became NaT — Pandas couldn't parse it without a year")

In [ ]:
# Extract useful date parts
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["day_name"] = df["order_date"].dt.day_name()
df["quarter"] = df["order_date"].dt.quarter

print("=== Extracted Date Parts ===")
print(df[["order_id", "order_date", "year", "month", "day_name", "quarter"]])

In [ ]:
# Date arithmetic — how many days ago was each order?
today = pd.Timestamp.today().normalize()
df["days_ago"] = (today - df["order_date"]).dt.days

print("=== Days Since Order ===")
print(df[["order_id", "order_date", "days_ago"]])

---
## 9. String Cleaning

Inconsistent strings ruin joins and groupbys.

In [ ]:
# Create some messy string data
messy = pd.DataFrame({
    "customer": ["  Rahim ", "fatima", "KARIM", " Rahim", "Fatima  "],
    "city":     ["Dhaka", "dhaka", " Chittagong ", "DHAKA", "chittagong"]
})
print("=== Messy Strings ===")
print(messy)
print()
print(f"Unique customers: {messy['customer'].nunique()}")
print(f"Unique cities: {messy['city'].nunique()}")
print("(Should be 3 customers and 2 cities!)")

In [ ]:
# Fix 1: Strip whitespace
messy["customer"] = messy["customer"].str.strip()
messy["city"] = messy["city"].str.strip()

# Fix 2: Standardize case — title case for names
messy["customer"] = messy["customer"].str.title()
messy["city"] = messy["city"].str.title()

print("=== After Cleaning ===")
print(messy)
print()
print(f"Unique customers: {messy['customer'].nunique()}")
print(f"Unique cities: {messy['city'].nunique()}")
print("✅ Now correct!")

In [ ]:
# Other useful string methods
sample = pd.Series(["Laptop Pro", "phone basic", "TABLET Max", "lap top"])

print("Lower:", sample.str.lower().tolist())
print("Upper:", sample.str.upper().tolist())
print("Title:", sample.str.title().tolist())
print()

# Replace values
fixed = sample.str.replace("lap top", "Laptop", case=False)
print("After replace:", fixed.tolist())
print()

# Check for patterns
has_pro = sample.str.contains("pro", case=False, na=False)
print("Contains 'pro':", has_pro.tolist())

---
## 10. Replace & Map Values

In [ ]:
# replace() — fix specific values
products = pd.Series(["Lap top", "mob", "Tablet", "Lap top", "mob"])

# Replace single value
fixed = products.replace("Lap top", "Laptop")
print("Single replace:", fixed.tolist())

# Replace multiple values at once
fixed = products.replace({"Lap top": "Laptop", "mob": "Phone"})
print("Multi replace: ", fixed.tolist())

In [ ]:
# map() — transform entire column using a dictionary
products = pd.Series(["Laptop", "Phone", "Tablet", "Laptop", "Phone"])

tier_map = {
    "Laptop": "Premium",
    "Phone": "Mid-Range",
    "Tablet": "Budget"
}

tiers = products.map(tier_map)
print("Product → Tier mapping:")
for p, t in zip(products, tiers):
    print(f"  {p:10s} → {t}")

In [ ]:
# map() with a function (lambda)
amounts = pd.Series([85000, 55000, 32000, 12000, 2500])

categories = amounts.map(lambda x: "High" if x > 50000 else "Medium" if x > 10000 else "Low")
print("Amount categories:")
for a, c in zip(amounts, categories):
    print(f"  {a:>6,} BDT → {c}")

---
## 11. Complete Cleaning Pipeline

Putting it all together — a reusable cleaning function.

In [ ]:
def clean_orders(df):
    """Standard cleaning pipeline for orders data"""
    df = df.copy()
    
    print(f"Starting rows: {len(df)}")

    # 1. Remove duplicates
    df = df.drop_duplicates(subset=["order_id"])
    print(f"After removing duplicates: {len(df)}")

    # 2. Handle nulls
    df["customer"] = df["customer"].fillna("Unknown")
    df["product"] = df["product"].fillna(df["product"].mode()[0])
    df = df.dropna(subset=["amount"])
    print(f"After handling nulls: {len(df)}")

    # 3. Fix data types
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    # 4. Standardize strings
    df["customer"] = df["customer"].str.strip().str.title()
    df["product"] = df["product"].str.strip().str.title()

    # 5. Reset index
    df = df.reset_index(drop=True)

    print(f"Final rows: {len(df)}")
    print("✅ Cleaning complete!")
    return df


# Run the pipeline
df_raw = pd.DataFrame(data)
df_clean = clean_orders(df_raw)
print()
print(df_clean)
print()
print(df_clean.dtypes)

---
## ☕ Hands-On Exercise 2: Full Cleaning Pipeline

Apply everything you've learned:

1. Start with the raw messy DataFrame
2. Remove duplicates based on order_id
3. Convert amount to numeric
4. Convert order_date to datetime
5. Fill remaining nulls
6. Standardize all string columns
7. Add a "month" column from order_date
8. Add a "spending_tier" column (High > 50000, Medium > 10000, Low)
9. Print `df.info()` to verify everything
10. Export to `orders_cleaned.csv`


In [ ]:
# ✅ Exercise 2 Solution

# Start fresh
df = pd.DataFrame(data)
print("=== Raw Data ===")
print(df)
print()

# 1. Remove duplicates
df = df.drop_duplicates(subset=["order_id"])
print(f"After removing duplicates: {len(df)} rows")

# 2. Convert amount to numeric
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# 3. Convert order_date to datetime
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

# 4. Fill nulls
df["customer"] = df["customer"].fillna("Unknown")
df["product"] = df["product"].fillna(df["product"].mode()[0])
df = df.dropna(subset=["amount"])

# 5. Standardize strings
df["customer"] = df["customer"].str.strip().str.title()
df["product"] = df["product"].str.strip().str.title()

# 6. Add month column
df["month"] = df["order_date"].dt.month

# 7. Add spending tier
conditions = [df["amount"] > 50000, df["amount"] > 10000]
choices = ["High", "Medium"]
df["spending_tier"] = np.select(conditions, choices, default="Low")

# 8. Reset index
df = df.reset_index(drop=True)

# 9. Verify
print()
print("=== Cleaned Data ===")
print(df)
print()
df.info()
print()
print("=== Spending Tiers ===")
print(df["spending_tier"].value_counts())

# 10. Export
df.to_csv("orders_cleaned.csv", index=False)
print("\n✅ Exported to orders_cleaned.csv")